# TP 2.1 — Exploration de l’annuaire avec PySpark**Donnée** : `donnees/fr-en-annuaire-education.csv` (séparateur `;`).**Sources** : `Docs/sources_data.md` — section TP 2.1.**Prérequis** : Java 11 ou 17 installé (`java -version`). Voir `README.md` à la racine du dépôt.

In [ ]:
from pathlib import Path

def resolve_data_dir() -> Path:
    """Racine du dépôt = dossier qui contient `donnees/`. Lancer Jupyter depuis cette racine."""
    cwd = Path.cwd()
    if (cwd / "donnees").is_dir():
        return cwd
    p = cwd
    for _ in range(4):
        if (p / "donnees").is_dir():
            return p
        p = p.parent
    return cwd

ROOT = resolve_data_dir()
DATA = ROOT / "donnees"
print("Répertoire données :", DATA.resolve())


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder.master("local[*]")
    .appName("TP21_Annuaire")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark


In [ ]:
csv_path = str(DATA / "fr-en-annuaire-education.csv")

df = spark.read.csv(
    csv_path,
    header=True,
    inferSchema=True,
    sep=";",
)

df.printSchema()
df.show(3, truncate=50)


In [ ]:
df.count()


In [ ]:
# Agrégations par type d'établissement (libelle_nature)
by_nature = df.groupBy("libelle_nature").count().orderBy(F.desc("count"))
by_nature.show(25, truncate=False)


In [ ]:
# Communes avec le plus d'établissements
by_commune = (
    df.groupBy("code_commune", "nom_commune")
    .count()
    .orderBy(F.desc("count"))
    .limit(15)
)
by_commune.show(truncate=False)


In [ ]:
# Lycées professionnels, publics, Île-de-France (libellé région sans accent dans le jeu)
filt = df.filter(
    (F.col("libelle_nature") == "LYCEE PROFESSIONNEL")
    & (F.col("statut_public_prive") == "Public")
    & (F.col("libelle_region") == "Ile-de-France")
)
print("Nombre de lignes :", filt.count())
filt.select("identifiant_de_l_etablissement", "nom_etablissement", "nom_commune").show(10, truncate=False)


## Comparaison de performance Spark vs pandas (comptage par `libelle_nature`)

In [ ]:
import time
import pandas as pd

t0 = time.perf_counter()
spark_counts = df.groupBy("libelle_nature").count().collect()
t_spark = time.perf_counter() - t0

pdf = pd.read_csv(DATA / "fr-en-annuaire-education.csv", sep=";", low_memory=False)
t1 = time.perf_counter()
_ = pdf.groupby("libelle_nature").size()
t_pandas = time.perf_counter() - t1

print(f"Spark  : {t_spark:.3f} s")
print(f"Pandas : {t_pandas:.3f} s")


In [ ]:
out_dir = str(DATA / "tp21_agregat_parquet")
(
    df.groupBy("libelle_nature")
    .count()
    .write.mode("overwrite")
    .parquet(out_dir)
)
print("Écrit :", out_dir)


In [ ]:
spark.stop()


## SynthèseDiscuter en groupe : quand Spark apporte-t-il un gain par rapport à pandas sur **cette** volumétrie ? À partir de quel ordre de grandeur de données la distribution devient-elle pertinente ?